In [3]:
# 内核须能 import aps（如 PYTHONPATH=Z:\pyagent）、OpenRouter 与 Logfire（可选 LOGFIRE_TOKEN）
from aps.core.logfire_setup import init_logfire
from aps.agents.orchestrator import OrchestratorAgent

init_logfire()

async def main():
    agent = OrchestratorAgent()
    text = "本周订单务必准时交付，可以少换几次产"
    out = await agent.run(text)
    print("user_intent:", out.user_intent)
    print("required_tasks:", out.required_tasks)
    print("strategy_hint:", out.strategy_hint)

await main()

Logfire project URL: https://logfire-us.pydantic.dev/evango/starter-project

22:49:01.218 agent run
22:49:01.221   chat xiaomi/mimo-v2-flash
user_intent: 用户希望确保本周订单准时交付，同时尽量减少生产换产次数
required_tasks: [<TaskType.PLAN: 'plan'>, <TaskType.SCHEDULE: 'schedule'>, <TaskType.EXPLAIN: 'explain'>]
strategy_hint: 优先交期，最小换产


In [4]:
from aps.agents.base import AgentContext
from aps.agents.planner import PlannerAgent

text = "本周订单务必准时交付，可以少换几次产"

ctx = AgentContext(
    orders_info="- O1: 产品A 500件, 截止24h\n- O2: 产品B 300件, 截止12h",
    machines_info="- M1(线1): 支持类型[beverage, dairy]\n- M2(线2): 支持类型[beverage, juice]",
)

async def try_planner():
    planner = PlannerAgent()
    out = await planner.run(text, ctx)
    print("strategy:", out.strategy)
    print("weights:", out.weights.model_dump())
    print("time_limit_seconds:", out.time_limit_seconds)
    print("allow_delays:", out.allow_delays, "max_delay_hours:", out.max_delay_hours)
    print("explanation:", out.explanation)
    params = out.to_optimization_params()
    print("OptimizationParams:", params.model_dump())

await try_planner()


22:49:04.835 agent run
22:49:04.837   chat xiaomi/mimo-v2-flash
strategy: OptimizationStrategy.ON_TIME_DELIVERY
weights: {'on_time': 0.6, 'changeover': 0.2, 'utilization': 0.1, 'profit': 0.1}
time_limit_seconds: 60
allow_delays: False max_delay_hours: 0.0
explanation: 用户明确要求"本周订单务必准时交付"，因此选择on_time策略，给予最高权重0.6。同时用户提到"可以少换几次产"，说明换产优化是次要目标，给予0.2权重。未提及产能利用率和利润目标，分别给予较低权重0.1。不允许延期以确保准时交付。
OptimizationParams: {'strategy': <OptimizationStrategy.ON_TIME_DELIVERY: 'on_time'>, 'time_limit_seconds': 60, 'weights': {'on_time': 0.6, 'changeover': 0.2, 'utilization': 0.1, 'profit': 0.1}, 'planning_horizon_hours': 168, 'allow_late_delivery': False, 'max_late_hours': 0}


In [7]:
# 七步：① Orchestrator ② Planner ③ Scheduler ④ Validator ⑤ Adjuster ⑥ Explainer ⑦ Monitor（LLM）
import json
from pathlib import Path
from aps.core.logfire_setup import init_logfire

init_logfire()
try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

# 从项目根或 aps 子目录加载 .env（内核 cwd 不一定是仓库根时也能读到）
for _env in (Path.cwd() / ".env", Path.cwd() / "aps" / ".env"):
    if _env.is_file() and load_dotenv:
        load_dotenv(_env, override=False)

from aps.agents.base import AgentContext
from aps.agents.orchestrator import APSSystem
from aps.tests.sample_scenario import get_sample_machines, get_sample_orders


async def main():
    user_text = "本周订单务必准时，尽量少换产"
    system = APSSystem(orders=get_sample_orders(), machines=get_sample_machines())

    # ① Orchestrator
    orch_out = await system.orchestrator_agent.run(user_text)
    print("① Orchestrator — user_intent:", orch_out.user_intent)
    print("   required_tasks:", orch_out.required_tasks)
    print("   strategy_hint:", orch_out.strategy_hint)

    # ② Planner（同一句话 + 订单/机器摘要）
    ctx = AgentContext(
        user_input=user_text,
        orders_info=system._format_orders(),
        machines_info=system._format_machines(),
    )
    planner_out = await system.planner.run(user_text, ctx)
    print("② Planner — strategy:", planner_out.strategy)
    print("   weights:", planner_out.weights.model_dump())
    print("   allow_delays:", planner_out.allow_delays, "max_delay_hours:", planner_out.max_delay_hours)
    params = planner_out.to_optimization_params()
    print("   OptimizationParams:", params.model_dump())

    # ③ SchedulerAgent.run_optimization（同步，与 test_scheduler_agent / APSSystem._execute_schedule 一致）
    schedule = system.scheduler.run_optimization(params)
    print("③ Scheduler — task_count:", schedule.task_count, "total_makespan:", round(schedule.total_makespan, 2))
    print("   on_time_delivery_rate:", schedule.on_time_delivery_rate, "delayed_count:", schedule.delayed_count)
    for i, a in enumerate(schedule.get_sorted_assignments()[:5], 1):
        print(f"   {i}. {a.machine_id} {a.product_name} {a.start_time:.1f}-{a.end_time:.1f}h")

    # ④ ValidatorAgent.validate（异步 + LLM，与 APSSystem._validate_result 一致）
    validation = await system.validator.validate(schedule)
    print("④ Validator — is_valid:", validation.is_valid)
    print("   quality_score:", validation.quality_score, "confidence:", validation.confidence_score)
    print("   overall_status:", validation.overall_status)
    print("   violations:", len(validation.constraint_violations), "warnings:", len(validation.warnings))
    if validation.warnings:
        print("   warnings:", validation.warnings[:5])
    if validation.recommendations:
        print("   recommendations:", validation.recommendations[:5])

    # ⑤ AdjusterAgent.analyze_and_adjust（与 APSSystem._adjust_if_needed 一致）
    # 当前实现：验证失败且存在 type==due_date 的违反项时会 handle_order_change 并重跑求解；
    # 不调用 Adjuster 内 pydantic_ai.Agent，一般不再产生新的 Logfire chat span。
    adjustment = await system.adjuster.analyze_and_adjust(
        schedule, validation, system.orders, system.machines
    )
    if adjustment is None:
        print("⑤ Adjuster — 无调整动作（验证通过，或无匹配的违反项策略）")
    else:
        print("⑤ Adjuster —", adjustment.action_type, "|", adjustment.reason)
        print("   affected_orders:", adjustment.affected_orders)
        print("   new_schedule_id:", adjustment.new_schedule_id, "changes:", adjustment.changes)

    # ⑥ ExplainAgent.run（异步 + LLM，与 APSSystem._explain_result 一致）
    explain_ctx = AgentContext(
        user_input=user_text,
        orders_info=system._format_orders(),
        machines_info=system._format_machines(),
        optimization_params=params.model_dump(),
        schedule_result=schedule.model_dump(),
    )
    explanation = await system.explainer.run(user_text, explain_ctx)
    print("⑥ ScheduleExplanation（完整）")
    print(json.dumps(explanation.model_dump(mode="json"), ensure_ascii=False, indent=2))

    # ⑦ MonitorAgent.run（异步 + LLM，与 APSSystem._monitor_result 一致）
    monitor_report = await system.monitor.run(schedule)
    print("⑦ MonitorReport（完整）")
    print(json.dumps(monitor_report.model_dump(mode="json"), ensure_ascii=False, indent=2))


await main()

23:56:32.839 agent run
23:56:32.841   chat xiaomi/mimo-v2-flash
① Orchestrator — user_intent: 用户要求本周订单务必准时交付，同时尽量减少生产换产次数
   required_tasks: [<TaskType.PLAN: 'plan'>, <TaskType.SCHEDULE: 'schedule'>, <TaskType.EXPLAIN: 'explain'>]
   strategy_hint: 优先交期，最小换产
23:56:35.626 agent run
23:56:35.627   chat xiaomi/mimo-v2-flash
② Planner — strategy: OptimizationStrategy.ON_TIME_DELIVERY
   weights: {'on_time': 0.5, 'changeover': 0.3, 'utilization': 0.1, 'profit': 0.1}
   allow_delays: True max_delay_hours: 24.0
   OptimizationParams: {'strategy': <OptimizationStrategy.ON_TIME_DELIVERY: 'on_time'>, 'time_limit_seconds': 60, 'weights': {'on_time': 0.5, 'changeover': 0.3, 'utilization': 0.1, 'profit': 0.1}, 'planning_horizon_hours': 168, 'allow_late_delivery': True, 'max_late_hours': 24}
③ Scheduler — task_count: 3 total_makespan: 11.0
   on_time_delivery_rate: 1.0 delayed_count: 0
   1. M002 可乐 0.0-6.0h
   2. M001 牛奶 0.0-5.0h
   3. M002 橙汁 6.0-11.0h
23:56:37.645 agent run
23:56:37.646   chat xi

In [ ]:
# --- Engine 层单测（无 LLM）：APSSolver / CP-SAT vs 启发式 ---
# 需内核能 import aps；与 tests/test_engine.py、SchedulerAgent 使用同一入口。

import json

from aps.engine.cp_sat_solver import HAS_ORTOOLS, CPSATSolver
from aps.engine.solver import APSSolver
from aps.models.optimization import OptimizationParams, OptimizationStrategy
from aps.tests.sample_scenario import get_sample_machines, get_sample_orders

orders = get_sample_orders()
machines = get_sample_machines()
params_on_time = OptimizationParams(
    strategy=OptimizationStrategy.ON_TIME_DELIVERY,
    time_limit_seconds=30,
)

print("HAS_ORTOOLS (可装 ortools 则 CP-SAT 可用):", HAS_ORTOOLS)

# 1) 默认路径：APSSolver（自动 CP-SAT 或启发式）
solver_default = APSSolver(orders=orders, machines=machines, params=params_on_time)
result_cp = solver_default.solve()
print("\n[APSSolver 默认] is_optimal:", result_cp.is_optimal, "planning_s:", round(result_cp.planning_time_seconds, 4))
print("  task_count:", result_cp.task_count, "makespan:", result_cp.total_makespan, "on_time_delivery_rate:", result_cp.on_time_delivery_rate)
for a in result_cp.get_sorted_assignments():
    print(f"  {a.machine_id} {a.product_name} {a.start_time:.1f}-{a.end_time:.1f}h on_time={a.is_on_time}")

# 2) 强制启发式（对照）
solver_heur = APSSolver(
    orders=orders, machines=machines, params=params_on_time, use_cp_sat=False
)
result_h = solver_heur.solve()
print("\n[APSSolver use_cp_sat=False 启发式] is_optimal:", result_h.is_optimal)
print("  makespan:", result_h.total_makespan, "on_time_delivery_rate:", result_h.on_time_delivery_rate)

# 3) 直接使用 CPSATSolver（与 engine 内部一致）
cp = CPSATSolver(orders=orders, machines=machines, params=params_on_time)
result_direct = cp.solve()
print("\n[CPSATSolver 直接] is_optimal:", result_direct.is_optimal, "tasks:", result_direct.task_count)

# 4) 可选：完整 JSON（便于复制对比）
# print(json.dumps(result_cp.model_dump(mode="json"), ensure_ascii=False, indent=2))
